#C2 M5 Demo 2 - Sequence Model for Character-Level Text Generation

###**Scenario:**
A publishing startup is developing an AI-assisted writing tool that helps authors by generating stylistically consistent character-level text, such as prose in the style of a given author (e.g., Shakespeare, Edgar Allan Poe, or J.K. Rowling). The tool needs to understand and reproduce character-level patterns like spelling, grammar, and authorial style.

The team wants a prototype that can generate plausible English text at the character level, trained on a corpus of a selected author. You are tasked with building this prototype using RNN-based sequence models, applying key techniques in character-level modeling.

###**Objectives:**
**Preprocess Character-Level Text Data**

* Tokenize characters, create input-output sequences, and apply padding/masking as needed.

**Explain RNN Architecture and Unrolling**

* Illustrate how RNNs process sequences step-by-step and maintain hidden states.

**Build and Train the RNN Model Using Keras**

* Implement a vanilla RNN or GRU with teacher forcing to generate character-level predictions.


**Address Core Training Challenges**

* Explain the vanishing gradient problem and how GRUs help mitigate it.

**Generate and Evaluate Text Output**

* Use a seed input to generate character sequences and evaluate stylistic consistency.

## Prepare Character-Level Text Data
Simulate a continuous plain-text corpus from structured entries and prepare it for character-level RNN training.

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("char_level_text_data.csv")
print(df.head())
print(f"Total samples: {len(df)}")

In [ ]:
text_corpus = ''.join(df['input_sequence'].tolist())
print(f"Corpus length: {len(text_corpus)} characters")
print("Sample:", text_corpus[:300])

In [ ]:
unique_chars = sorted(set(text_corpus))
print("Unique characters in the corpus:")
print(unique_chars)
print(f"Total unique characters: {len(unique_chars)}")

In [ ]:
char_to_idx = {char: idx for idx, char in enumerate(unique_chars)}
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

In [ ]:
encoded_text = np.array([char_to_idx[char] for char in text_corpus])
print(f"Encoded text sample: {encoded_text[:50]}")

In [ ]:
seq_length = 30
sequences = []
next_chars = []

for i in range(len(encoded_text) - seq_length):
    sequences.append(encoded_text[i:i+seq_length])
    next_chars.append(encoded_text[i+seq_length])

X = np.array(sequences)
y = np.array(next_chars)
print("Input shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

## Define and Build an RNN Architecture
Construct a character-level text generation model using a simple RNN or GRU.
We'll use Keras to define a sequential model with an embedding layer, an RNN (or GRU), and a dense output layer.

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense

In [ ]:
vocab_size = len(char_to_idx)
embedding_dim = 64
rnn_units = 128

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=seq_length),
    GRU(units=rnn_units, return_sequences=False),
    Dense(units=vocab_size, activation='softmax')
])

In [ ]:
model.build(input_shape=(None, seq_length)) # Use None for the batch size dimension

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
layer_names = [layer.name for layer in model.layers]

In [ ]:
param_counts = [layer.count_params() for layer in model.layers]

plt.figure(figsize=(8,4))
plt.bar(layer_names, param_counts, color='skyblue')
plt.title('Number of Parameters per Layer')
plt.ylabel('Parameter Count')
plt.xlabel('Layer')
plt.show()

## Train the Model Using Teacher Forcing

Train the RNN model on the prepared sequences. Teacher forcing is inherently used in sequence models during training by providing the correct previous character as input for the next prediction. We will fit the model using the training data.

In [ ]:
import matplotlib.pyplot as plt

batch_size = 64
epochs = 20

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=batch_size,
    epochs=epochs
)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

##Address the Vanishing Gradient Problem using GRUs vs. Vanilla RNNs

 Demonstrate how GRUs help mitigate the vanishing gradient problem compared to vanilla RNNs by building and comparing two simple models.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, GRU, Dense
import matplotlib.pyplot as plt

vocab_size = len(char_to_idx)
embedding_dim = 64
rnn_units = 128

In [ ]:
model_rnn = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=seq_length),
    SimpleRNN(rnn_units, return_sequences=False),
    Dense(vocab_size, activation='softmax')
])
model_rnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model_gru = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=seq_length),
    GRU(rnn_units, return_sequences=False),
    Dense(vocab_size, activation='softmax')
])
model_gru.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
print("Vanilla RNN Model Summary:")
model_rnn.summary()

print("\nGRU Model Summary:")
model_gru.summary()

In [ ]:
epochs = 5
batch_size = 64

history_rnn = model_rnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size, verbose=0)
history_gru = model_gru.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size, verbose=0)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(history_rnn.history['loss'], label='Vanilla RNN Train Loss')
plt.plot(history_rnn.history['val_loss'], label='Vanilla RNN Val Loss')
plt.plot(history_gru.history['loss'], label='GRU Train Loss')
plt.plot(history_gru.history['val_loss'], label='GRU Val Loss')
plt.title('Training and Validation Loss: Vanilla RNN vs GRU')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

##Implement Tokenization, Padding, and Masking for Variable-Length Sequences

 Prepare character-level sequences for training by converting characters to integer tokens, padding sequences to a fixed length, and applying masking to ignore padded tokens during training.

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Masking
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense
import matplotlib.pyplot as plt
import random

# Generate variable-length sequences (random lengths between 10 and 40)
max_seq_len = 40
variable_sequences = []
next_chars_var = []

In [ ]:
for i in range(len(encoded_text) - max_seq_len - 1):
    seq_len = random.randint(10, max_seq_len)
    seq = encoded_text[i:i+seq_len]
    target = encoded_text[i+seq_len]
    variable_sequences.append(seq)
    next_chars_var.append(target)

In [ ]:
padded_sequences = pad_sequences(variable_sequences, maxlen=max_seq_len, padding='pre', value=0)
next_chars_var = np.array(next_chars_var)

print(f"Padded sequences shape: {padded_sequences.shape}")
print(f"Targets shape: {next_chars_var.shape}")

In [ ]:
seq_lengths = [len(seq) for seq in variable_sequences]
plt.hist(seq_lengths, bins=range(10, max_seq_len+2), alpha=0.7, color='skyblue', edgecolor='black')
plt.title('Distribution of Original Sequence Lengths')
plt.xlabel('Sequence Length')
plt.ylabel('Frequency')
plt.show()


In [ ]:
model_masking = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_seq_len, mask_zero=True),
    GRU(units=rnn_units),
    Dense(vocab_size, activation='softmax')
])

model_masking.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_masking.summary()

In [ ]:
history_masking = model_masking.fit(
    padded_sequences, next_chars_var,
    validation_split=0.1,
    epochs=15,
    batch_size=64
)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history_masking.history['loss'], label='Train Loss')
plt.plot(history_masking.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
plt.subplot(1,2,2)
plt.plot(history_masking.history['accuracy'], label='Train Accuracy')
plt.plot(history_masking.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()